# TEST — Silver 1 Clean
Standalone test version of `silver1/clean.ipynb`.  
All parameters are hardcoded below — just edit and **Run All**.

In [ ]:
# ============================================================
# TEST PARAMETERS — edit these to match your test scenario
# ============================================================
source_name   = "fedex_direct_invoice"
bronze_path   = "bronze/fedex/invoice/"
silver1_table = "staging_fedex_direct_invoice"
file_format   = "csv"          # csv | excel | parquet | json | avro
extra_params  = {
    "date_format": "%m/%d/%Y",
    # "sheet": 0,              # uncomment for Excel
    # "amount_cols": [],       # columns to extract numeric values from
    # "date_cols": [],         # columns to parse as dates
}

WORKSPACE   = "dev-fabric-data"
LAKEHOUSE   = "fabric_lakehouse"
DW_ENDPOINT = "8dac972c-0ff1-400f-bba3-e1e2302070b5.datawarehouse.fabric.microsoft.com"
DW_DATABASE = "fabric_gold_warehouse"

In [ ]:
import io
import sys
from functools import reduce

from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F

FORMAT_GLOB = {
    "csv":     "*.csv",
    "excel":   "*.xlsx",
    "parquet": "*.parquet",
    "json":    "*.json",
    "avro":    "*.avro",
}
file_glob = FORMAT_GLOB.get(file_format, f"*.{file_format}")

### Shared lib & Config

In [ ]:
sys.path.insert(0, "/lakehouse/default/Files/shared")
import shared_functions as gf

run_id = "test-manual"

print(f"[TEST silver1/clean] source={source_name} table={silver1_table}")

spark = SparkSession.builder.getOrCreate()

### 1. List unprocessed bronze files

In [ ]:
with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
    records = gf.query_to_records(
        conn,
        f"SELECT file_name FROM data_ops.pipeline_run_log "
        f"WHERE source_name = '{source_name}' AND layer = 'silver1' AND status = 'success'"
    )
    processed_files = {r["file_name"] for r in records}

bronze_files = gf.list_files(
    workspace_name=WORKSPACE,
    lakehouse_name=LAKEHOUSE,
    prefix=bronze_path,
    pattern=file_glob,
)

new_files = [f for f in bronze_files if f.split("/")[-1] not in processed_files]
print(f"  Bronze files: {len(bronze_files)} total, {len(new_files)} unprocessed")

if not new_files:
    print("  Nothing to process. Stopping here.")

### 2. Read each file into a Spark DataFrame

In [ ]:
all_frames = []

for abfss_path in new_files:
    file_name = abfss_path.split("/")[-1]
    print(f"  Reading {file_name} ({file_format}) ...")

    if file_format == "csv":
        sdf = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(abfss_path)
        )
    elif file_format == "parquet":
        sdf = spark.read.parquet(abfss_path)
    elif file_format == "excel":
        import pandas as pd
        raw_bytes = gf.read_file_bytes(abfss_path)
        pdf = pd.read_excel(io.BytesIO(raw_bytes), sheet_name=extra_params.get("sheet", 0))
        sdf = spark.createDataFrame(pdf)
    elif file_format == "json":
        sdf = spark.read.option("multiline", True).json(abfss_path)
    elif file_format == "avro":
        sdf = spark.read.format("avro").load(abfss_path)
    else:
        raise ValueError(f"Unsupported file_format: '{file_format}'")

    sdf = (
        sdf
        .withColumn("_source_name", F.lit(source_name))
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_run_id",      F.lit(run_id))
    )
    all_frames.append(sdf)

if all_frames:
    combined = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), all_frames)
    print(f"  Raw rows: {combined.count()}, columns: {len(combined.columns)}")
else:
    print("  No frames loaded")

### 3. Clean

In [ ]:
if all_frames:
    # 3a: normalise column names, trim strings
    combined = gf.spark_clean_dataframe(combined)

    # 3b: extract numeric values from currency/percent string columns
    amount_cols = extra_params.get("amount_cols", [])
    if amount_cols:
        combined = gf.spark_extract_numeric(combined, amount_cols)
        print(f"  Extracted numeric: {amount_cols}")

    # 3c: parse date columns
    date_cols   = extra_params.get("date_cols", [])
    date_format = extra_params.get("date_format", None)
    if date_cols:
        combined = gf.spark_parse_dates(combined, date_cols, fmt=date_format)
        print(f"  Parsed dates: {date_cols} (fmt={date_format or 'auto'})")

    # 3d: coerce to existing Delta schema if table exists
    silver_abfss = gf.onelake_abfss(WORKSPACE, LAKEHOUSE, f"Tables/silver/{silver1_table}")
    try:
        existing_schema = spark.read.format("delta").load(silver_abfss).schema
        table_exists = True
        combined = gf.spark_cast_to_schema(combined, existing_schema)
        print(f"  Cast to existing Delta schema ({len(existing_schema.fields)} columns)")
    except Exception:
        table_exists = False

    # 3e: drop all-null rows
    data_cols = [c for c in combined.columns if not c.startswith("_")]
    combined = combined.filter(
        sum(F.col(c).isNotNull().cast("int") for c in data_cols) > 0
    )

    rows_written = combined.count()
    print(f"  After cleaning: {rows_written} rows")

### 4. Write to Silver Delta table

In [ ]:
if all_frames:
    if not table_exists:
        print(f"  Creating new Delta table at {silver_abfss}")
        (
            combined.write
            .format("delta")
            .mode("overwrite")
            .save(silver_abfss)
        )
    else:
        combined.createOrReplaceTempView("incoming")
        spark.sql(f"""
            MERGE INTO delta.`{silver_abfss}` AS target
            USING incoming AS source
            ON target._source_file = source._source_file
               AND target._source_name = source._source_name
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"  MERGE complete into {silver_abfss}")

### 5. Log

In [ ]:
if all_frames:
    with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
        for file_name in [f.split("/")[-1] for f in new_files]:
            gf.log_run(
                conn,
                source_name=source_name,
                layer="silver1",
                status="success",
                rows_processed=rows_written,
                run_id=run_id,
            )

    print(f"  Done: {rows_written} rows -> {silver1_table}")